# Seleção de features: filtro, wrapper, embutido, Boruta e SHAP

Escolher coluna é a decisão que vem depois de criar coluna. Existem três famílias clássicas, e duas
ferramentas que resolvem a parte difícil:

- **filtro**: mede cada coluna sozinha e corta pelas melhores (`SelectKBest`, correlação);
- **wrapper**: treina o modelo muitas vezes, tirando e pondo colunas, e fica com as que ajudam (`RFE`);
- **embutido**: o próprio modelo entrega a importância (`L1`, importância de árvore, permutation
  importance);
- **Boruta**: procura **todas** as colunas que carregam informação, não só as melhores k;
- **SHAP**: mede quanto cada coluna contribui para as previsões de um modelo treinado.

A diferença entre os dois grupos é a pergunta. Filtro, wrapper e embutido buscam o conjunto **mínimo que
basta** (minimal-optimal). Boruta busca **tudo o que carrega informação** (all-relevant), que costuma ser
a pergunta de quem quer entender o fenômeno em vez de só prever.

Como sempre nos notebooks desta pasta, a tabela tem verdade declarada: **6 colunas que importam**, **4
redundantes** (combinações lineares das que importam) e **14 de ruído puro**. Dá para medir quem acertou.


In [ ]:
%pip install boruta==0.4.3 shap==0.52.0


In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from boruta import BorutaPy
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import (RFE, RFECV, SequentialFeatureSelector, SelectKBest,
                                       f_classif, mutual_info_classif)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", message=".*will be removed.*")
sns.set_theme(context="notebook", style="whitegrid", palette="colorblind",
              rc={"figure.figsize": (9, 4)})

RANDOM_STATE = 42
ALVO = "cancelou"


def tabela_com_verdade(n_linhas=900, seed=RANDOM_STATE):
    """6 colunas que importam, 4 redundantes (combinação linear das que importam) e 14 de ruído."""
    rng = np.random.default_rng(seed)
    efeitos = [1.2, 1.0, 0.9, 0.7, 0.5, 0.4]
    informativas = {f"info_{i}": rng.normal(0, 1, n_linhas) for i in range(1, 7)}
    sinal = sum(e * informativas[f"info_{i}"] for i, e in enumerate(efeitos, start=1))
    sinal = sinal + rng.normal(0, 1.0, n_linhas)

    redundantes = {
        "red_1": 0.8 * informativas["info_1"] + 0.6 * informativas["info_2"] + rng.normal(0, 0.1, n_linhas),
        "red_2": 0.5 * informativas["info_3"] - 0.7 * informativas["info_4"] + rng.normal(0, 0.1, n_linhas),
        "red_3": 1.1 * informativas["info_5"] + rng.normal(0, 0.1, n_linhas),
        "red_4": 0.9 * informativas["info_6"] + rng.normal(0, 0.1, n_linhas),
    }
    ruido = {f"ruido_{i:02d}": rng.normal(0, 1, n_linhas) for i in range(1, 15)}

    tabela = pd.DataFrame({**informativas, **redundantes, **ruido})
    tabela[ALVO] = (sinal > np.quantile(sinal, 0.78)).astype(int)
    return tabela


tabela = tabela_com_verdade()
INFORMATIVAS = [f"info_{i}" for i in range(1, 7)]
REDUNDANTES = [f"red_{i}" for i in range(1, 5)]
RUIDO = [f"ruido_{i:02d}" for i in range(1, 15)]
COLUNAS = INFORMATIVAS + REDUNDANTES + RUIDO

X = tabela[COLUNAS]
y = tabela[ALVO]

print(f"tabela: {tabela.shape[0]} linhas x {len(COLUNAS)} colunas")
print(f"alvo: {y.mean():.1%} de cancelamento")
print(f"verdade: {len(INFORMATIVAS)} informativas, {len(REDUNDANTES)} redundantes, {len(RUIDO)} de ruido")


## A verdade, vista na correlação

Antes de comparar métodos, uma olhada no que a correlação de cada coluna com o alvo diz. As informativas
têm efeito direto; as redundantes têm efeito **herdado** das informativas que elas copiam; o ruído não tem
efeito nenhum.


In [ ]:
relacao = X.corrwith(y).abs().sort_values(ascending=False)
cor = pd.DataFrame({"coluna": COLUNAS})
cor["grupo"] = np.where(cor["coluna"].isin(INFORMATIVAS), "informativa",
                        np.where(cor["coluna"].isin(REDUNDANTES), "redundante", "ruido"))
cor["|correlacao| com o alvo"] = cor["coluna"].map(relacao).round(3)
cor = cor.sort_values("|correlacao| com o alvo", ascending=False)

print("as 10 colunas mais correlacionadas:")
print(cor.head(10).to_string(index=False))
print()
print("resumo por grupo:")
print(cor.groupby("grupo")["|correlacao| com o alvo"].agg(["mean", "min", "max"]).round(3).to_string())

sns.barplot(cor, x="|correlacao| com o alvo", y="coluna", hue="grupo", dodge=False)
plt.title("correlacao de cada coluna com o alvo, por grupo")
plt.ylabel("")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


A correlação separa bem informativa de ruído e mal informativa de redundante. E ela erra para cima: a
`red_1` aparece com 0,518, **acima** da `info_1`, que tem 0,389. Copiar uma coluna informativa com
folga faz a cópia parecer mais importante que o original, e é exatamente aí que filtro univariado tropeça.

## Filtro: mede cada coluna sozinha

Filtro é o mais barato e o mais cego: cada coluna é avaliada isoladamente, sem modelo nenhum. `f_classif`
usa teste de variância entre grupos, `mutual_info_classif` captura dependência que não é linear, e a
correlação é a versão manual da mesma ideia.


In [ ]:
def selecao_do_filtro():
    selecoes = {}
    t0 = time.perf_counter()
    for nome, funcao in {"f_classif": f_classif, "mutual_info": mutual_info_classif}.items():
        t = time.perf_counter()
        escolhidas = SelectKBest(funcao, k=10).fit(X, y).get_support()
        selecoes[f"filtro: SelectKBest {nome}"] = (list(X.columns[escolhidas]), "filtro",
                                                   time.perf_counter() - t)
    t = time.perf_counter()
    selecoes["filtro: correlacao (top 10)"] = (list(relacao.head(10).index), "filtro",
                                               time.perf_counter() - t)
    print(f"filtro inteiro: {time.perf_counter() - t0:.3f}s")
    return selecoes


selecoes = selecao_do_filtro()
for nome, (colunas, _, tempo) in selecoes.items():
    print(f"  {nome:32s} {len(colunas):2d} colunas em {tempo:.3f}s")


## Wrapper: o modelo decide, tentando

Wrapper treina o modelo várias vezes, tirando e pondo colunas, e usa a nota dele para decidir. É mais
caro e costuma acertar mais, porque a escolha é feita pelo que o modelo realmente usa: `RFE` tira a pior
de cada vez, `RFECV` decide sozinho quantas ficam, e `SequentialFeatureSelector` vai somando a melhor de
cada vez.

## Embutido: o modelo entrega a importância

Aqui não tem laço nenhum: o modelo é treinado uma vez e a importância sai dele. Na regressão logística
com penalidade L1, o coeficiente de quem não ajuda vai a zero. Na floresta, a importância vem da impureza.
E `permutation_importance` embaralha uma coluna de cada vez e mede o estrago.


In [ ]:
def selecao_do_wrapper_e_embutido():
    selecoes = {}
    modelo_base = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
    # o RFE precisa saber onde esta o coeficiente, ja que o estimador e um Pipeline
    onde_esta_o_coeficiente = "named_steps.logisticregression.coef_"

    t = time.perf_counter()
    escolhidas = RFE(modelo_base, n_features_to_select=10,
                     importance_getter=onde_esta_o_coeficiente).fit(X, y).get_support()
    selecoes["wrapper: RFE (10)"] = (list(X.columns[escolhidas]), "wrapper", time.perf_counter() - t)

    t = time.perf_counter()
    escolhidas = RFECV(modelo_base, cv=3, scoring="roc_auc", min_features_to_select=2,
                       importance_getter=onde_esta_o_coeficiente).fit(X, y).get_support()
    selecoes["wrapper: RFECV (k automatico)"] = (list(X.columns[escolhidas]), "wrapper", time.perf_counter() - t)

    t = time.perf_counter()
    escolhidas = SequentialFeatureSelector(modelo_base, n_features_to_select=10, direction="forward",
                                           cv=3, scoring="roc_auc").fit(X, y).get_support()
    selecoes["wrapper: SFS adiante (10)"] = (list(X.columns[escolhidas]), "wrapper", time.perf_counter() - t)

    t = time.perf_counter()
    l1 = make_pipeline(StandardScaler(), LogisticRegression(l1_ratio=1.0, solver="saga", C=0.5,
                                                           max_iter=5000)).fit(X, y)
    coeficientes = np.abs(l1[-1].coef_[0])
    selecoes["embutido: L1 (coeficiente != 0)"] = (list(X.columns[coeficientes > 1e-8]), "embutido",
                                                  time.perf_counter() - t)

    t = time.perf_counter()
    floresta = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X, y)
    selecoes["embutido: importancia da arvore"] = (
        list(pd.Series(floresta.feature_importances_, index=X.columns).nlargest(10).index), "embutido",
        time.perf_counter() - t)

    t = time.perf_counter()
    permutacao = permutation_importance(floresta, X, y, n_repeats=10, random_state=RANDOM_STATE,
                                       scoring="roc_auc")
    selecoes["embutido: permutation importance"] = (
        list(pd.Series(permutacao.importances_mean, index=X.columns).nlargest(10).index), "embutido",
        time.perf_counter() - t)
    return selecoes


selecoes.update(selecao_do_wrapper_e_embutido())
for nome, (colunas, _, tempo) in selecoes.items():
    if nome.startswith(("wrapper", "embutido")):
        print(f"  {nome:34s} {len(colunas):2d} colunas em {tempo:.2f}s")


## Boruta: procurar todas as relevantes

Boruta embaralha cada coluna para criar uma cópia "sombra", treina a floresta nas colunas reais mais as
sombra, e compara: coluna que não bate a melhor sombra é ruído. O que sobra é **confirmado**, o que fica
na dúvida é **tentativa**, e o resto é **rejeitado**.

A pergunta dele é outra: não é "quais são as 10 melhores", é "**quais carregam informação**". Uma coluna
redundante pode sobreviver, porque o critério é uso e não novidade: aqui o Boruta confirmou 3 das 4
redundantes, e isso não é defeito, é a diferença entre as perguntas.


In [ ]:
t = time.perf_counter()
boruta = BorutaPy(RandomForestClassifier(n_estimators=300, max_depth=5, random_state=RANDOM_STATE),
                  n_estimators="auto", perc=100, alpha=0.05, max_iter=100, random_state=RANDOM_STATE)
boruta.fit(X.to_numpy(), y.to_numpy())
tempo_boruta = time.perf_counter() - t

confirmadas = list(X.columns[boruta.support_])
tentativas = list(X.columns[(~boruta.support_) & boruta.support_weak_])
selecoes["Boruta (confirmadas)"] = (confirmadas, "all-relevant", tempo_boruta)

print(f"tempo: {tempo_boruta:.1f}s")
print(f"confirmadas ({len(confirmadas)}): {confirmadas}")
print(f"tentativas ({len(tentativas)}): {tentativas}")
print()
print("as confirmadas sao verdadeiras?")
for grupo, colunas in (("informativas", INFORMATIVAS), ("redundantes", REDUNDANTES), ("ruido", RUIDO)):
    print(f"  de {len(colunas):2d} {grupo:12s} confirmadas: {len(set(confirmadas) & set(colunas))}")


## SHAP: quanto cada coluna contribui

SHAP responde outra pergunta: dado **este** modelo treinado, quanto cada coluna empurrou a previsão de
cada linha. Ele não diz se a coluna é relevante no mundo, diz quanto ela foi usada.

E tem um efeito colateral que importa aqui: com duas colunas correlacionadas, o crédito se divide entre
elas. É o que acontece entre `info_5` e `red_3`, que são quase a mesma coluna.


In [ ]:
import shap

floresta_final = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X, y)
t = time.perf_counter()
valores = shap.TreeExplainer(floresta_final).shap_values(X)
matriz_shap = valores[:, :, 1] if valores.ndim == 3 else valores
importancia_shap = (pd.Series(np.abs(matriz_shap).mean(axis=0), index=X.columns)
                    .sort_values(ascending=False))
selecoes["SHAP (top 10)"] = (list(importancia_shap.head(10).index), "contribuicao",
                            time.perf_counter() - t)

print("as 12 colunas que o SHAP mais usou, com o grupo de cada uma:")
for coluna, valor in importancia_shap.head(12).items():
    grupo = ("informativa" if coluna in INFORMATIVAS else
             "redundante" if coluna in REDUNDANTES else "ruido")
    print(f"  {coluna:10s} {valor:.4f}  {grupo}")
print()
print(f"credito somado das 6 informativas: {importancia_shap[INFORMATIVAS].sum():.3f}")
print(f"credito somado das 4 redundantes:  {importancia_shap[REDUNDANTES].sum():.3f}")
print(f"credito somado das 14 de ruido:    {importancia_shap[RUIDO].sum():.3f}")

sns.barplot(x=importancia_shap.head(12).values, y=importancia_shap.head(12).index)
plt.title("|SHAP| medio, as 12 colunas mais usadas pelo modelo")
plt.xlabel("")
plt.tight_layout()
plt.show()


## A comparação, método por método

Agora a tabela que o notebook existe para produzir. Para cada método: quantas colunas escolheu, quantas
das 6 informativas pegou, quantas das 4 redundantes, quantos falsos positivos de ruído, e a nota de uma
validação cruzada usando **só** as colunas escolhidas, no mesmo modelo e na mesma partição.


In [ ]:
def auc_medio(colunas):
    if not colunas:
        return float("nan")
    cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
    modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
    return cross_val_score(modelo, X[list(colunas)], y, cv=cv, scoring="roc_auc").mean()


linhas = [{
    "método": "todas as 24 colunas (sem selecao)", "família": "-", "selecionadas": len(COLUNAS),
    "das 6 informativas": len(INFORMATIVAS), "das 4 redundantes": len(REDUNDANTES),
    "ruído (falsos positivos)": len(RUIDO), "ROC-AUC (CV-5)": round(auc_medio(COLUNAS), 4),
    "tempo (s)": 0.0,
}]
for nome, (colunas, familia, tempo) in selecoes.items():
    escolhidas = set(colunas) & set(COLUNAS)
    linhas.append({
        "método": nome, "família": familia, "selecionadas": len(escolhidas),
        "das 6 informativas": len(escolhidas & set(INFORMATIVAS)),
        "das 4 redundantes": len(escolhidas & set(REDUNDANTES)),
        "ruído (falsos positivos)": len(escolhidas & set(RUIDO)),
        "ROC-AUC (CV-5)": round(auc_medio(sorted(escolhidas)), 4),
        "tempo (s)": round(tempo, 3),
    })

comparacao = pd.DataFrame(linhas).set_index("método").sort_values("ROC-AUC (CV-5)", ascending=False)
display(comparacao)


Leia a tabela com calma, porque o resultado é menos óbvio do que parece:

- **sem seleção nenhuma, a nota já é 0,9510** (24 colunas, 14 delas de ruído), e o melhor método chega a
  0,9568. Todos os métodos ficam dentro de 0,04, e os sete melhores dentro de 0,006. Nesta tabela, com
  900 linhas e sinal forte, a seleção **não compra nota**: compra coluna. Menos coluna é menos coleta,
  menos manutenção e menos coisa para quebrar, e isso é ganho real em produção;
- **o Boruta foi o único que pegou zero coluna de ruído** (confirmou 9: as 6 informativas e 3
  redundantes), e pagou por isso com o maior tempo, 17,6 s contra 0,013 s do filtro;
- **o filtro de `mutual_info` foi o pior de todos**, 0,9198, e pegou só 4 das 6 informativas trazendo 4
  de ruído. Método mais "inteligente" no papel, pior na medição: é a razão de medir em vez de supor;
- **o `SFS` ficou com 3 informativas, 3 redundantes e 4 de ruído**, e ainda assim empatou com o resto em
  0,9558: nota alta não prova que a seleção foi boa, prova que o problema é fácil;
- **o L1 com `C=0.5` podou pouco**: ficou com 20 colunas, 13 delas de ruído. Regularização fraca não é
  seleção.

E o resultado que amarra o notebook: o que separa os métodos não é a nota, é **quantas colunas de ruído
cada um deixou passar**, que é a coluna mais informativa da tabela.


In [ ]:
matriz = pd.DataFrame(0, index=list(selecoes), columns=COLUNAS)
for nome, (colunas, _, _) in selecoes.items():
    matriz.loc[nome, [c for c in colunas if c in COLUNAS]] = 1

matriz = matriz[COLUNAS]
sns.heatmap(matriz, cbar=False, cmap=sns.color_palette(["#f0f0f0", "#2b6cb0"], as_cmap=True),
            linewidths=0.5, linecolor="white")
plt.axvline(6, color="black", linewidth=1.5)
plt.axvline(10, color="black", linewidth=1.5)
plt.title("o que cada metodo escolheu: informativas | redundantes | ruido")
plt.xlabel("")
plt.tight_layout()
plt.show()


## Dados reais

Em tabela de verdade não existe verdade declarada, e é aí que a comparação muda de sentido: o que se
ganha é menos coluna para o mesmo resultado. Aqui o Boruta nas 30 colunas do câncer de mama.


In [ ]:
dados = load_breast_cancer()
X_cancer = pd.DataFrame(dados.data, columns=dados.feature_names)
y_cancer = pd.Series(dados.target, name="maligno")

t = time.perf_counter()
boruta_cancer = BorutaPy(RandomForestClassifier(n_estimators=300, max_depth=5,
                                                random_state=RANDOM_STATE),
                         n_estimators="auto", max_iter=100, random_state=RANDOM_STATE)
boruta_cancer.fit(X_cancer.to_numpy(), y_cancer.to_numpy())
confirmadas_cancer = list(X_cancer.columns[boruta_cancer.support_])

cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
todas = cross_val_score(modelo, X_cancer, y_cancer, cv=cv, scoring="roc_auc").mean()
poucas = cross_val_score(modelo, X_cancer[confirmadas_cancer], y_cancer, cv=cv, scoring="roc_auc").mean()

print(f"tempo: {time.perf_counter() - t:.1f}s")
print(f"confirmadas pelo Boruta: {len(confirmadas_cancer)} de {X_cancer.shape[1]}")
print(f"roc_auc com as 30 colunas:      {todas:.4f}")
print(f"roc_auc com as confirmadas:     {poucas:.4f}")
print(f"diferenca: {poucas - todas:+.4f}")


## Como ler isso

1. escreva a pergunta antes de escolher o método: **conjunto mínimo que basta** (filtro, wrapper,
   embutido) ou **tudo o que carrega informação** (Boruta)?
2. filtro é barato e bom para tirar ruído óbvio, e ruim para coluna redundante, que ele mantém;
3. wrapper custa caro e acerta mais, porque mede o que o modelo usa;
4. embutido é o meio termo, e `permutation_importance` costuma ser o mais honesto dos três quando o
   modelo não tem importância própria;
5. SHAP explica o modelo treinado, não a relevância da coluna, e divide o crédito entre correlacionadas;
6. **faça a seleção dentro do pipeline** quando o número de colunas for decidido pela mesma medição, senão
   a escolha vê o teste;
7. menos coluna pelo mesmo resultado é ganho real: menos manutenção, menos coleta, menos coisa para
   quebrar.

## Resumo

Seleção de features é uma decisão com cinco ferramentas e duas perguntas. Neste notebook, com verdade
declarada, dá para dizer quem errou, e o erro mais comum não é o que se espera: **a nota quase não se
moveu**. Sem seleção, 0,9510; com o melhor método, 0,9568. O que mudou de verdade foi quantas colunas de
ruído cada um deixou passar: 14 sem seleção, 1 ou 2 nos métodos bons, e zero no Boruta, que é o único
all-relevant e o mais caro.

Também ficou medido o que a intuição erra: a correlação põe a coluna redundante acima da informativa que
ela copia; o `mutual_info` perdeu para o `f_classif`; o `SFS` acertou menos informativas que o filtro e
empatou na nota; e o SHAP deu o maior crédito justamente à `red_1`, que é cópia. **Menos coluna pelo
mesmo resultado é o ganho**, e em tabela de verdade é o único que se pode prometer.

**Próximo passo:** o notebook de desbalanceamento desta pasta, que é o caso em que a escolha de coluna e a
escolha da métrica se misturam.

## Referências

```bibtex
@article{guyon2003introduction,
  title   = {An Introduction to Variable and Feature Selection},
  author  = {Guyon, Isabelle and Elisseeff, Andr{'e}},
  journal = {Journal of Machine Learning Research},
  volume  = {3},
  pages   = {1157--1182},
  year    = {2003}
}

@article{kursa2010boruta,
  title   = {Feature Selection with the Boruta Package},
  author  = {Kursa, Miron B. and Rudnicki, Witold R.},
  journal = {Journal of Statistical Software},
  volume  = {36},
  number  = {11},
  pages   = {1--13},
  year    = {2010},
  doi     = {10.18637/jss.v036.i11}
}

@inproceedings{lundberg2017unified,
  title     = {A Unified Approach to Interpreting Model Predictions},
  author    = {Lundberg, Scott M. and Lee, Su-In},
  booktitle = {Advances in Neural Information Processing Systems (NeurIPS)},
  volume    = {30},
  year      = {2017}
}
```
